# Data Integration and Feature Engineering

## 1. Introduction

This notebook integrates client and property datasets and transforms the combined information into a client-level feature dataset suitable for machine learning-based buyer segmentation.

The property dataset contains multiple records for individual clients. Therefore, property-level information is aggregated to create meaningful client-level behavioral and investment features.

The resulting dataset will be prepared through categorical encoding and feature scaling before applying clustering algorithms.

------------------

## 2. Import Libraries

The required libraries are imported for data loading, manipulation, feature engineering, and preprocessing.

In [21]:
import pandas as pd
import numpy as np

## 3. Load the Datasets

The cleaned client and property datasets are loaded into Pandas DataFrames.

The client dataset provides demographic and behavioral characteristics, while the property dataset provides transaction and property-level information.

In [22]:
clients_df = pd.read_csv('../data/Clients.csv')
property_df = pd.read_csv('../data/Properties.csv')

## Inspection of both datasets before integrating it

In [23]:
print("Client IDs:")
print(clients_df['client_id'].head(10))

print("\nProperty Client References:")
print(property_df['client_ref'].dropna().head(10))

Client IDs:
0    C0001
1    C0002
2    C0003
3    C0004
4    C0005
5    C0006
6    C0007
7    C0008
8    C0009
9    C0010
Name: client_id, dtype: str

Property Client References:
0    C0027
1    C0097
2    C0113
3    C0141
4    C0146
5    C0023
6    C0040
7    C0007
8    C0082
9    C0038
Name: client_ref, dtype: str


In [24]:
print("Unique clients:", clients_df['client_id'].nunique())
print("Unique property client references:", property_df['client_ref'].nunique())

Unique clients: 2000
Unique property client references: 2000


## Checking any unmatched property client reference

In [25]:
property_df['client_ref'].value_counts().describe()

count    2000.0000
mean        3.6525
std         0.8397
min         3.0000
25%         3.0000
50%         4.0000
75%         4.0000
max        13.0000
Name: count, dtype: float64

## 4. Integrate Client and Property Data

The client and property datasets are linked using the client identifier.

A client may be associated with multiple property records. Therefore, the merged dataset may contain multiple rows for the same client at this stage.

A left join is used with the client dataset as the primary table so that every client remains represented in the integrated dataset.

In [26]:
print("Property shape:", property_df.shape)
print("Client shape:", clients_df.shape)

Property shape: (10000, 9)
Client shape: (2000, 12)


In [27]:
merged_df = clients_df.merge(
    property_df,
    how = 'left',
    right_on = 'client_ref',
    left_on = 'client_id',
    
)

### 4.1 Validate Client References

The property client references are compared with the client IDs to identify property records that refer to clients not present in the client dataset.

Missing `client_ref` values are excluded from this comparison because they represent property records without a recorded client reference rather than invalid client IDs.

In [28]:
property_client_ids = set(property_df['client_ref'].dropna())
client_ids = set(clients_df['client_id'])

unmatched_ids = property_client_ids - client_ids

print("Unmatched property client references:", len(unmatched_ids))

Unmatched property client references: 0


In [29]:
merged_df.shape

(7305, 21)

In [30]:
merged_df.head(10)

,client_id,client_type,first_name,last_name,date_of_birth,gender,country,region,acquisition_purpose,satisfaction_score,...,referral_channel,listing_id,tower_number,transaction_date,unit_category,unit_number,floor_area_sqft,sale_price,listing_status,client_ref
0,C0001,Individual,Kareem,Liu,05-11-1968,F,USA,California,Home,4,...,Website,90343,9,10-01-2024,Apartment,40,1090.32,"$351,419.29",Sold,C0001
1,C0001,Individual,Kareem,Liu,05-11-1968,F,USA,California,Home,4,...,Website,4051,4,12-01-2024,Apartment,51,1608.84,"$496,266.41",Sold,C0001
2,C0001,Individual,Kareem,Liu,05-11-1968,F,USA,California,Home,4,...,Website,150099,15,05-01-2025,Apartment,15,522.71,"$175,599.90",Sold,C0001
3,C0001,Individual,Kareem,Liu,05-11-1968,F,USA,California,Home,4,...,Website,30432,3,12-01-2025,Apartment,50,713.67,"$223,479.12",Sold,C0001
4,C0002,Individual,Trystan,Oconnor,11/26/1962,M,USA,California,Home,1,...,Website,150044,15,01-01-2024,Apartment,6,938.57,"$299,245.20",Sold,C0002
5,C0002,Individual,Trystan,Oconnor,11/26/1962,M,USA,California,Home,1,...,Website,1045,1,02-01-2024,Apartment,45,756.21,"$248,525.12",Sold,C0002
6,C0002,Individual,Trystan,Oconnor,11/26/1962,M,USA,California,Home,1,...,Website,90285,9,12-01-2024,Apartment,36,1582.79,"$505,127.63",Sold,C0002
7,C0002,Individual,Trystan,Oconnor,11/26/1962,M,USA,California,Home,1,...,Website,200450,20,05-01-2025,Apartment,51,1062.33,"$336,892.39",Sold,C0002
8,C0002,Individual,Trystan,Oconnor,11/26/1962,M,USA,California,Home,1,...,Website,30377,3,12-01-2025,Apartment,46,1599.81,"$451,305.59",Sold,C0002
9,C0003,Individual,Kale,Gay,04-07-1959,M,USA,California,Home,4,...,Agency,10104,1,07-01-2024,Apartment,13,719.47,"$216,874.97",Sold,C0003


## 5. Data Type Conversion and Cleaning

The integrated dataset contains some numerical and date-related fields stored as text.

These variables are converted into appropriate data types so that mathematical calculations and aggregation can be performed correctly.

In [31]:
merged_df[['floor_area_sqft', 'sale_price', 'transaction_date']].info()

<class 'pandas.DataFrame'>
RangeIndex: 7305 entries, 0 to 7304
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   floor_area_sqft   7305 non-null   float64
 1   sale_price        7305 non-null   str    
 2   transaction_date  7305 non-null   str    
dtypes: float64(1), str(2)
memory usage: 322.9 KB


In [32]:
merged_df[['floor_area_sqft', 'sale_price', 'transaction_date']].head()

,floor_area_sqft,sale_price,transaction_date
0,1090.32,"$351,419.29",10-01-2024
1,1608.84,"$496,266.41",12-01-2024
2,522.71,"$175,599.90",05-01-2025
3,713.67,"$223,479.12",12-01-2025
4,938.57,"$299,245.20",01-01-2024


### 5.1 Convert Sale Price to Numeric

The `sale_price` values contain currency symbols and comma separators.

These non-numeric characters are removed before converting the values to floating-point numbers.

In [33]:
merged_df['sale_price'] = (
    merged_df['sale_price']
    .str.replace('$', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(float)
)

### 5.2 Convert Transaction Date

The `transaction_date` column is converted from text into Pandas datetime format.

This allows the transaction dates to be used correctly for temporal analysis and future feature engineering.

In [34]:
merged_df['transaction_date'] = pd.to_datetime(
    merged_df['transaction_date'],
    format='mixed',
    dayfirst=False,
    errors='coerce'
)

In [35]:
merged_df[['floor_area_sqft', 'sale_price', 'transaction_date']].info()

<class 'pandas.DataFrame'>
RangeIndex: 7305 entries, 0 to 7304
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   floor_area_sqft   7305 non-null   float64       
 1   sale_price        7305 non-null   float64       
 2   transaction_date  7305 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(2)
memory usage: 171.3 KB


In [36]:
merged_df[['floor_area_sqft', 'sale_price', 'transaction_date']].head()

,floor_area_sqft,sale_price,transaction_date
0,1090.32,351419.29,2024-10-01
1,1608.84,496266.41,2024-12-01
2,522.71,175599.90,2025-05-01
3,713.67,223479.12,2025-12-01
4,938.57,299245.20,2024-01-01


In [37]:
merged_df[['sale_price', 'floor_area_sqft']].describe()

,sale_price,floor_area_sqft
count,7305.000000,7305.000000
mean,345072.000115,1141.147838
std,132053.768307,419.241645
min,97402.800000,410.710000
25%,232448.630000,782.150000
50%,330997.320000,1110.880000
75%,451237.670000,1501.640000
max,736652.270000,1957.160000


## 6. Client-Level Feature Engineering

The integrated data contains multiple property records per client.

To perform buyer segmentation, property-level information is aggregated so that each client is represented by a single row.

The following features are created to capture purchasing behavior and investment scale:

### 6.1 Property Count

`property_count` represents the number of property records associated with each client.

This feature can help distinguish clients with relatively few purchases from clients with repeated or larger-scale property activity.

In [38]:
property_count = (
    merged_df.groupby('client_id')['listing_id']
    .count()
    .reset_index(name='property_count')
)

property_count.head()

,client_id,property_count
0,C0001,4
1,C0002,5
2,C0003,5
3,C0004,6
4,C0005,13


### 6.2 Total Property Value

`total_property_value` represents the combined sale value of all linked properties for each client.

This feature is used as a proxy for the overall scale of a client's property activity.

Because the client dataset does not contain an income variable, this measure should not be interpreted as the client's income. Instead, it represents the value of their observed property transactions.

In [39]:
total_property_value = (
    merged_df.groupby('client_id')['sale_price']
    .sum()
    .reset_index(name='total_property_value')
)

total_property_value.head()

,client_id,total_property_value
0,C0001,1246764.72
1,C0002,1841095.93
2,C0003,1661457.59
3,C0004,1608263.51
4,C0005,3653385.38


### 6.3 Average Property Value

`average_property_value` represents the mean sale price of the properties associated with each client.

This feature helps identify differences in the typical property value associated with different buyers.

In [40]:
average_property_value = (
    merged_df.groupby('client_id')['sale_price']
    .mean()
    .reset_index(name='average_property_value')
)

average_property_value.head()

,client_id,average_property_value
0,C0001,311691.180000
1,C0002,368219.186000
2,C0003,332291.518000
3,C0004,268043.918333
4,C0005,281029.644615


### 6.4 Average Floor Area

`average_floor_area` represents the average size of properties associated with each client.

It provides an additional measure of the type and scale of properties purchased by different buyers.

In [41]:
average_floor_area = (
    merged_df.groupby('client_id')['floor_area_sqft']
    .mean()
    .reset_index(name='average_floor_area')
)

average_floor_area.head()

,client_id,average_floor_area
0,C0001,983.885000
1,C0002,1187.942000
2,C0003,1058.110000
3,C0004,937.103333
4,C0005,927.296154


## 7. Combine Property-Level Features

The property-level features created above are combined into a single client-level feature dataset using `client_id` as the common key.

This allows the different measures of property activity to be analyzed together for each client.

In [42]:
client_features = property_count.merge(
    total_property_value,
    on='client_id',
    how='left'
)

client_features.head()

,client_id,property_count,total_property_value
0,C0001,4,1246764.72
1,C0002,5,1841095.93
2,C0003,5,1661457.59
3,C0004,6,1608263.51
4,C0005,13,3653385.38


### 7.1 Add Average Property Value

The `average_property_value` feature is merged with the existing client-level property features using `client_id`.

This adds information about the typical value of properties associated with each client.

In [70]:
client_features = client_features.merge(
    average_property_value,
    on='client_id',
    how='left'
)

### 7.2 Add Average Floor Area

The `average_floor_area` feature is merged into the client-level feature dataset using `client_id`.

This completes the set of aggregated property features, providing information about the number, value, and average size of properties associated with each client.

In [71]:
client_features = client_features.merge(
    average_floor_area,
    on='client_id',
    how='left'
)

In [44]:
client_features.head()

,client_id,property_count,total_property_value,average_property_value,average_floor_area
0,C0001,4,1246764.72,311691.180000,983.885000
1,C0002,5,1841095.93,368219.186000,1187.942000
2,C0003,5,1661457.59,332291.518000,1058.110000
3,C0004,6,1608263.51,268043.918333,937.103333
4,C0005,13,3653385.38,281029.644615,927.296154


In [45]:
client_features.shape

(2000, 5)

In [46]:
client_features.isnull().sum()

client_id                 0
property_count            0
total_property_value      0
average_property_value    0
average_floor_area        0
dtype: int64

## 8. Create Client Age Feature

The `date_of_birth` column is converted from a text format into a datetime format to enable age calculation.

A reference date is then used to calculate the age of each client. Age provides a more useful demographic feature for buyer segmentation than the original date of birth.

In [47]:
clients_df['date_of_birth'] = pd.to_datetime(
    clients_df['date_of_birth'],
    format='mixed',
    dayfirst=False,
    errors='coerce'
)

reference_date = pd.Timestamp('2026-01-01')

clients_df['age'] = (
    (reference_date - clients_df['date_of_birth']).dt.days // 365
)

### 8.1 Validate Age

The derived `age` feature is examined to verify that the calculated ages are reasonable and that the conversion did not produce invalid or missing values.

In [74]:
clients_df[['age']].head()

,age
0,57
1,63
2,66
3,66
4,49


## 9. Select Relevant Client Features

Relevant demographic and behavioral variables are selected from the client dataset for buyer segmentation.

Personal identifiers such as `first_name` and `last_name` are excluded because they do not provide meaningful information about buyer characteristics. The original `date_of_birth` is also excluded because the derived `age` feature provides a more suitable numerical representation for clustering.

In [75]:
client_info = clients_df[
    [
        'client_id',
        'client_type',
        'gender',
        'country',
        'region',
        'acquisition_purpose',
        'satisfaction_score',
        'loan_applied',
        'referral_channel',
        'age'
    ]
]

client_info.head()

,client_id,client_type,gender,country,region,acquisition_purpose,satisfaction_score,loan_applied,referral_channel,age
0,C0001,Individual,F,USA,California,Home,4,Yes,Website,57
1,C0002,Individual,M,USA,California,Home,1,No,Website,63
2,C0003,Individual,M,USA,California,Home,4,Yes,Agency,66
3,C0004,Individual,M,USA,California,Home,5,No,Website,66
4,C0005,Company,M,USA,California,Investment,5,No,Website,49


## 9.1 Combine Client and Property Features

The selected client characteristics are combined with the aggregated property features to create a complete client-level dataset.

The features are merged using `client_id` as the common identifier. A left join is used so that all clients in the client dataset remain included in the final dataset.

The resulting dataset combines demographic and behavioral characteristics with property-related measures such as property count, total property value, average property value, and average floor area.

In [55]:
client_features = client_info.merge(
    property_count,
    on='client_id',
    how='left'
)

client_features = client_features.merge(
    total_property_value,
    on='client_id',
    how='left'
)

client_features = client_features.merge(
    average_property_value,
    on='client_id',
    how='left'
)

client_features = client_features.merge(
    average_floor_area,
    on='client_id',
    how='left'
)

### 9.2 Validate the Combined Dataset

The dimensions and missing values of the resulting client-level feature dataset are checked to ensure that the merging process was completed correctly.

The expected result is one row per client with the relevant client and property features.

In [76]:
client_features.shape

(2000, 16)

In [78]:
client_features.isnull().sum()

client_id                   0
client_type                 0
gender                      0
country                     0
region                      0
acquisition_purpose         0
satisfaction_score          0
loan_applied                0
referral_channel            0
age                         0
property_count              0
total_property_value        0
average_property_value_x    0
average_floor_area_x        0
average_property_value_y    0
average_floor_area_y        0
dtype: int64

In [56]:
client_features.head()

,client_id,client_type,gender,country,region,acquisition_purpose,satisfaction_score,loan_applied,referral_channel,age,property_count,total_property_value,average_property_value,average_floor_area
0,C0001,Individual,F,USA,California,Home,4,Yes,Website,57,4,1246764.72,311691.180000,983.885000
1,C0002,Individual,M,USA,California,Home,1,No,Website,63,5,1841095.93,368219.186000,1187.942000
2,C0003,Individual,M,USA,California,Home,4,Yes,Agency,66,5,1661457.59,332291.518000,1058.110000
3,C0004,Individual,M,USA,California,Home,5,No,Website,66,6,1608263.51,268043.918333,937.103333
4,C0005,Company,M,USA,California,Investment,5,No,Website,49,13,3653385.38,281029.644615,927.296154


In [58]:
client_features.columns

Index(['client_id', 'client_type', 'gender', 'country', 'region',
       'acquisition_purpose', 'satisfaction_score', 'loan_applied',
       'referral_channel', 'age', 'property_count', 'total_property_value',
       'average_property_value', 'average_floor_area'],
      dtype='str')

## 10. Inspect the Final Feature Dataset

Descriptive statistics are examined for the numerical features in the final client-level dataset.

This helps verify the ranges and distributions of the demographic and property-related variables before applying machine learning preprocessing.

In [80]:
client_features.describe()

,satisfaction_score,age,property_count,total_property_value,average_property_value_x,average_floor_area_x,average_property_value_y,average_floor_area_y
count,"2,000.00","2,000.00","2,000.00","2,000.00","2,000.00","2,000.00","2,000.00","2,000.00"
mean,3.03,55.15,3.65,"1,260,375.48","347,089.96","1,147.48","347,089.96","1,147.48"
std,1.41,17.36,0.84,"347,830.66","69,721.82",219.92,"69,721.82",219.92
min,1.00,25.00,3.00,"463,611.95","154,537.32",564.01,"154,537.32",564.01
25%,2.00,40.00,3.00,"1,025,238.01","295,807.68",984.95,"295,807.68",984.95
50%,3.00,56.00,4.00,"1,220,893.17","341,523.38","1,129.18","341,523.38","1,129.18"
75%,4.00,70.00,4.00,"1,441,973.88","390,843.32","1,296.56","390,843.32","1,296.56"
max,5.00,94.00,13.00,"3,653,385.38","563,423.50","1,800.45","563,423.50","1,800.45"


### 10.1 Feature Data Types

The data types of the final features are examined to distinguish numerical variables from categorical variables.

This distinction is necessary because numerical and categorical variables require different preprocessing techniques before clustering.

In [60]:
client_features.dtypes

client_id                     str
client_type                   str
gender                        str
country                       str
region                        str
acquisition_purpose           str
satisfaction_score          int64
loan_applied                  str
referral_channel              str
age                         int64
property_count              int64
total_property_value      float64
average_property_value    float64
average_floor_area        float64
dtype: object

### Dropping the 'client_id' since it will not play a important role in ml

In [63]:
ml_features = client_features.drop(columns=['client_id'])

ml_features.head()

,client_type,gender,country,region,acquisition_purpose,satisfaction_score,loan_applied,referral_channel,age,property_count,total_property_value,average_property_value,average_floor_area
0,Individual,F,USA,California,Home,4,Yes,Website,57,4,"1,246,764.72","311,691.18",983.88
1,Individual,M,USA,California,Home,1,No,Website,63,5,"1,841,095.93","368,219.19","1,187.94"
2,Individual,M,USA,California,Home,4,Yes,Agency,66,5,"1,661,457.59","332,291.52","1,058.11"
3,Individual,M,USA,California,Home,5,No,Website,66,6,"1,608,263.51","268,043.92",937.10
4,Company,M,USA,California,Investment,5,No,Website,49,13,"3,653,385.38","281,029.64",927.30


### 11.1 Identify Categorical Features

The categorical variables are examined to determine the number of unique categories in each feature.

This helps assess the dimensionality that will result from one-hot encoding before the features are passed to the clustering algorithm.

In [65]:
for col in [
    'client_type',
    'gender',
    'country',
    'region',
    'acquisition_purpose',
    'loan_applied',
    'referral_channel'
]:
    print(f"\n{col}:")
    print(ml_features[col].nunique(), "unique values")


client_type:
2 unique values

gender:
2 unique values

country:
10 unique values

region:
57 unique values

acquisition_purpose:
2 unique values

loan_applied:
2 unique values

referral_channel:
3 unique values


## 12. Categorical Feature Encoding

K-Means clustering requires numerical input, while several buyer characteristics in the dataset are categorical.

One-hot encoding is used to convert categorical variables into binary numerical features. Each category is represented by a separate column containing 0 or 1.

This allows categorical characteristics such as client type, country, region, acquisition purpose, loan behavior, and referral channel to be incorporated into the clustering process.

In [66]:
categorical_columns = [
    'client_type',
    'gender',
    'country',
    'region',
    'acquisition_purpose',
    'loan_applied',
    'referral_channel'
]

encoded_features = pd.get_dummies(
    ml_features,
    columns=categorical_columns,
    drop_first=False,
    dtype=int
)

encoded_features.head()

,satisfaction_score,age,property_count,total_property_value,average_property_value,average_floor_area,client_type_Company,client_type_Individual,gender_F,gender_M,...,region_Western Australia,region_Wyoming,region_Zealand,acquisition_purpose_Home,acquisition_purpose_Investment,loan_applied_No,loan_applied_Yes,referral_channel_Agency,referral_channel_Client,referral_channel_Website
0,4,57,4,"1,246,764.72","311,691.18",983.88,0,1,1,0,...,0,0,0,1,0,0,1,0,0,1
1,1,63,5,"1,841,095.93","368,219.19","1,187.94",0,1,0,1,...,0,0,0,1,0,1,0,0,0,1
2,4,66,5,"1,661,457.59","332,291.52","1,058.11",0,1,0,1,...,0,0,0,1,0,0,1,1,0,0
3,5,66,6,"1,608,263.51","268,043.92",937.10,0,1,0,1,...,0,0,0,1,0,1,0,0,0,1
4,5,49,13,"3,653,385.38","281,029.64",927.30,1,0,0,1,...,0,0,0,0,1,1,0,0,0,1


In [67]:
encoded_features.shape

(2000, 84)

### 12.1 Encoding Result

After one-hot encoding, the dataset contains 2,000 clients and 84 numerical features.

The increase in the number of columns is expected because each category in the categorical variables is represented by a separate binary feature.

## 13. Feature Scaling

The features in the dataset have different numerical ranges. For example, `satisfaction_score` ranges from 1 to 5, `age` ranges from 25 to 94, while `total_property_value` contains values in the millions.

Since K-Means clustering uses distance to measure similarity between clients, features with larger numerical values could have a greater influence on the clustering results.

To prevent this, StandardScaler is used to standardize the features so that they are placed on a comparable scale.

In [68]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

scaled_features = scaler.fit_transform(encoded_features)

### 13.1 Standardization Using StandardScaler

`StandardScaler` transforms the features so that each feature has a mean close to 0 and a standard deviation close to 1.

The `fit_transform()` method first learns the statistical properties of the features and then applies the standardization.

The original `encoded_features` DataFrame is not modified. The standardized values are stored separately in `scaled_features`.

In [69]:
scaled_features.shape

(2000, 84)

### 13.2 Final Feature Matrix

After categorical encoding and feature scaling, the dataset contains 2,000 clients represented by 84 standardized numerical features.

This feature matrix is now ready for the clustering stage. The next step is to determine the appropriate number of buyer segments before applying the K-Means algorithm.